# zynarai — ACE-Step 1.5 Inference (both adapters)

Run generation with **both** of your adapters active:
- **DiT LoKr adapter** = your *sound* (sub-bass, production) — loaded in the UI's LoRA panel
- **Merged 0.6B groove-LM** = your *composition* — selected as the 5Hz LM planner

Prereqs on Drive (`lora fine tune/`): the DiT LoRA at `output/lokr_v1/checkpoints/...`
and the merged LM at `output/acestep-5Hz-lm-0.6B-zynarai/` (from the LM notebook, cell 10).

Runtime: L4 GPU. Run cells top to bottom, then follow the UI steps in the last cell.

In [ ]:
# 1 — Clone ACE-Step + deps + meta-tensor patch (~5 min first run)
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
![ -d ACE-Step-1.5 ] || git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git
%cd /content/ACE-Step-1.5
lines = open('requirements.txt').read().splitlines()
SKIP = ('torch==','torchvision==','torchaudio==','flash-attn','--extra-index-url','triton')
open('/tmp/req.txt','w').write('\n'.join(l for l in lines if not any(t in l for t in SKIP)))
!pip install -q -r /tmp/req.txt
!pip install -q loguru lycoris-lora vector_quantize_pytorch
import os, re, vector_quantize_pytorch
vq_dir = os.path.dirname(vector_quantize_pytorch.__file__)
for fname in os.listdir(vq_dir):
    if fname.endswith('.py'):
        p = os.path.join(vq_dir, fname); t = open(p).read()
        t2 = re.sub(r'assert \(([^)]+?) > 1\)\.all\(\)', r"assert \1.device.type == 'meta' or (\1 > 1).all()", t)
        if t2 != t: open(p,'w').write(t2)
print('setup done')

In [ ]:
# 2 — Drive + checkpoints + your merged groove-LM
from google.colab import drive; drive.mount('/content/drive')
import os
from huggingface_hub import snapshot_download
if not os.path.exists('/content/loradrive'):
    os.symlink('/content/drive/MyDrive/lora fine tune', '/content/loradrive')
CKPT = '/content/checkpoints'
snapshot_download('ACE-Step/Ace-Step1.5', local_dir=CKPT)
snapshot_download('ACE-Step/acestep-v15-base', local_dir=f'{CKPT}/acestep-v15-base')
snapshot_download('ACE-Step/acestep-5Hz-lm-0.6B', local_dir=f'{CKPT}/acestep-5Hz-lm-0.6B')
# link the merged groove-LM from Drive into the dir the UI scans
merged_drive = '/content/loradrive/output/acestep-5Hz-lm-0.6B-zynarai'
merged_link  = f'{CKPT}/acestep-5Hz-lm-0.6B-zynarai'
if os.path.exists(merged_drive) and not os.path.exists(merged_link):
    os.symlink(merged_drive, merged_link)
elif not os.path.exists(merged_drive):
    print('WARNING: merged LM not found on Drive — run the LM notebook cell 10 first')
# the UI scans <repo>/checkpoints
if not os.path.exists('/content/ACE-Step-1.5/checkpoints'):
    os.symlink(CKPT, '/content/ACE-Step-1.5/checkpoints')
print('5Hz LMs available:', [d for d in os.listdir(CKPT) if d.startswith('acestep-5Hz')])

In [ ]:
# 3 — List your DiT LoRA epoch checkpoints (copy a path for the UI's LoRA panel)
import os
dit_ckpt_dir = '/content/loradrive/output/lokr_v1/checkpoints'
if os.path.isdir(dit_ckpt_dir):
    for d in sorted(os.listdir(dit_ckpt_dir)):
        if os.path.exists(f'{dit_ckpt_dir}/{d}/lokr_weights.safetensors'):
            print(f'{dit_ckpt_dir}/{d}')
else:
    print('DiT checkpoints not found at', dit_ckpt_dir)

In [ ]:
# 4 — Launch the Gradio UI (public link). MUST run from the repo dir so `acestep` imports.
%cd /content/ACE-Step-1.5
import os
os.environ['ACESTEP_CHECKPOINT_DIR'] = '/content/checkpoints'
!python -m acestep.acestep_v15_pipeline --share

## In the Gradio UI (from the `*.gradio.live` link)

**1. Service Configuration → initialize:**
```
Main Model Path   = acestep-v15-base
5Hz LM Model Path = acestep-5Hz-lm-0.6B-zynarai   ← your merged groove-LM
VAE = official | Device = auto | uncheck Compile Model | Initialize 5Hz LM = ON
→ click Load / Initialize
```

**2. LoRA panel (expand the collapsed 'LoRA' accordion) → load your DiT sound:**
```
LoRA path = <paste an epoch path from cell 3>
click  Load LoRA   (the BUTTON — toggling the checkbox alone does nothing)
confirm lora_status shows it loaded (272 LokrModules / DoRA)
tick use LoRA | LoRA scale = 0.5   (1.0 clips)
```

**3. Generate:** steps 50, shift 1.0, fixed seed.

## The A/B tests (same seed each pair)
```
isolate the LM (grooves):  swap 5Hz LM between  acestep-5Hz-lm-0.6B-zynarai  and  acestep-5Hz-lm-0.6B
                           (DiT LoRA fixed) → difference = composition/arrangement
isolate the DiT (sound):   toggle the LoRA multiplier 0.5 vs 0.0
                           (LM fixed) → difference = timbre/production
full stack vs base:        both on  vs  base LM + no DiT LoRA
```
Listen for: does 'both on' finally read as **your tracks** — your sound AND your grooves?